# initialization

In [1]:
# Local path to gluten jar.
gluten_target_jar='/opt/gluten/jars/gluten-velox-bundle.jar'

# Select workload. Can be either 'tpch' or 'tpcds'.
workload='tpcds'

# Run with gluten. If False, run Spark.
run_gluten=True

# TPC tables
#tpch_tabledir='/opt/spark/database/tpch_sf10_parquet_zstd'
#tpcds_tabledir='/opt/spark/database/tpch_sf10_parquet_zstd'

tabledir = 's3a://presto-workload/tpcds_sf500_parquet_zstd'

# Database name. if it's set, use the database instead loading table from tabledir
database = ''

# TPC queries
#tpch_query_path='/opt/spark/tpch-queries'
#tpcds_query_path='/opt/spark/tpcds-queries'

tpc_query_path = '/incubator-gluten/tools/gluten-it/common/src/main/resources/tpcds-queries'

# Parallelism
executors_per_node=16

#gluten_tpch_task_per_core=2
#gluten_tpcds_task_per_core=4
#spark_tpch_task_per_core=8
#spark_tpcds_task_per_core=8

task_per_core=2

# Offheap ratio. 0 to disable offheap for Spark.
# onheap:offheap = 1:2
#spark_offheap_ratio=2.0
# onheap:offheap = 1:7
#gluten_offheap_ratio=7.0

offheap_ratio = 7.0

# spark.io.compression.codec
spark_codec='lz4'
# spark.gluten.sql.columnar.shuffle.codec
gluten_codec='lz4'

In [2]:
%env PYSPARK_SUBMIT_ARGS=--driver-java-options -Dio.netty.tryReflectionSetAccessible=true --conf spark.executor.extraJavaOptions=-Dio.netty.tryReflectionSetAccessible=true pyspark-shell

env: PYSPARK_SUBMIT_ARGS=--driver-java-options -Dio.netty.tryReflectionSetAccessible=true --conf spark.executor.extraJavaOptions=-Dio.netty.tryReflectionSetAccessible=true pyspark-shell


In [3]:
%run /opt/spark/work-dir/ipython/native_sql_initialize.ipynb

home: /home/spark
cwd: /opt/spark/work-dir/ipython


,Environment Variable,Value
0,HOSTNAME,ip-10-167-37-206.ec2.internal
1,SHARED_LIBS,/shared-libs
2,JAVA_HOME,/usr/lib/jvm/java-17-openjdk
3,VCPKG_BINARY_SOURCES,"clear;files,/var/cache/vcpkg,readwrite"
4,PWD,/opt/spark/work-dir
5,HOME,/home/spark
6,VCPKG_PATH,/var/cache/vcpkg
7,CCACHE_DIR,/root/.cache/ccache
8,GCC_TOOL,/opt/rh/gcc-toolset-11
9,SPARK_HOME,/opt/spark


localhost: ip-10-167-37-206.ec2.internal
ip: 10.167.37.206
Spark version from SPARK_HOME: 3.5.2


# Application Level Configuration

In [4]:
if run_gluten:
    sct=GlutenSparkContext(executors_per_node, task_per_core, gluten_target_jar, offheap_ratio)
else:
    sct=VanillaSparkContext(executors_per_node, task_per_core, gluten_target_jar, offheap_ratio)

In [5]:
if workload.lower()=="tpch":
    bm=TPCHBenchmark(sct, tabledir, 'parquet', tpc_query_path)
else:
    bm=TPCDSBenchmark(sct, tabledir, 'parquet', tpc_query_path)

sct.conf.set('spark.hadoop.fs.s3a.aws.credentials.provider','org.apache.hadoop.fs.s3a.auth.IAMInstanceCredentialsProvider')

if run_gluten:
    sct.conf.set('spark.gluten.sql.columnar.shuffle.codec', gluten_codec)
    sct.conf.set('spark.gluten.sql.columnar.backend.velox.cacheEnabled', False)
    sct.conf.set('spark.gluten.sql.columnar.backend.velox.IOThreads', 48)
    sct.conf.set('spark.gluten.sql.columnar.backend.velox.loadQuantum', 268435456)
    sct.conf.set('spark.gluten.sql.columnar.backend.velox.maxCoalescedBytes', 67108864)
    sct.conf.set('spark.gluten.sql.columnar.backend.velox.maxCoalescedDistanceBytes', 1048576)
    sct.conf.set('spark.gluten.sql.columnar.backend.velox.memCacheSize', 134217728)
    sct.conf.set('spark.gluten.sql.columnar.backend.velox.prefetchRowGroups', 1)
    sct.conf.set('spark.gluten.sql.columnar.backend.velox.SplitPreloadPerDriver', 16)
    sct.conf.set('spark.gluten.sql.columnar.backend.velox.ssdCacheSize', 0)
    sct.conf.set('spark.locality.wait', 0)

else:
    sct.conf.set('spark.io.compression.codec', spark_codec)
    
bm.initialize()


            executors per node: 16
            parallelism: 32
            executor memory: 1024m
            offheap memory: 7168m
        
spark.serializer:  org.apache.spark.serializer.KryoSerializer
master:  spark://ip-10-167-37-206.ec2.internal:7077


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/08/15 07:03:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
W20250815 07:04:00.143832 111635 MemoryArbitrator.cpp:84] Query memory capacity[12.75GB] is set for NOOP arbitrator which has no capacity enforcement
/opt/spark/python/pyspark/sql/context.py:113: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


appid:  app-20250815070400-0006
SparkConf:


,key,value
0,spark.driver.host,ip-10-167-37-206.ec2.internal
1,spark.gluten.sql.columnar.backend.velox.cacheEnabled,False
2,spark.driver.extraClassPath,/opt/gluten/jars/gluten-velox-bundle.jar
3,spark.executor.extraClassPath,/opt/gluten/jars/gluten-velox-bundle.jar
4,spark.executor.instances,16
5,spark.gluten.sql.columnar.shuffle.codec,lz4
6,spark.executor.extraJavaOptions,-Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/jdk.internal.ref=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-opens=java.base/sun.util.calendar=ALL-UNNAMED --add-opens=java.security.jgss/sun.security.krb5=ALL-UNNAMED -Djdk.reflect.useDirectMethodHandle=false -Dio.netty.tryReflectionSetAccessible=true
7,spark.cleaner.periodicGC.interval,10s
8,spark.serializer,org.apache.spark.serializer.KryoSerializer
9,spark.master,spark://ip-10-167-37-206.ec2.internal:7077


start run:  app-20250815070400-0006
http://10.167.37.206:18080/history/app-20250815070400-0006/jobs/


In [6]:
bm.test_tpc.load_all_tables_as_tempview()

Loading all tables: ['call_center', 'catalog_page', 'catalog_returns', 'catalog_sales', 'customer', 'customer_address', 'customer_demographics', 'date_dim', 'household_demographics', 'income_band', 'inventory', 'item', 'promotion', 'reason', 'ship_mode', 'store', 'store_returns', 'store_sales', 'time_dim', 'warehouse', 'web_page', 'web_returns', 'web_sales', 'web_site']


In [7]:
sct.spark.sql('''
select count(*) h8_30_to_9
 from store_sales, household_demographics , time_dim, store
 where ss_sold_time_sk = time_dim.t_time_sk   
     and ss_hdemo_sk = household_demographics.hd_demo_sk 
     and ss_store_sk = s_store_sk
     and time_dim.t_hour = 8
     and time_dim.t_minute >= 30
     and ((household_demographics.hd_dep_count = 3 and household_demographics.hd_vehicle_count<=3+2) or
          (household_demographics.hd_dep_count = 0 and household_demographics.hd_vehicle_count<=0+2) or
          (household_demographics.hd_dep_count = 1 and household_demographics.hd_vehicle_count<=1+2)) 
     and store.s_store_name = 'ese'
''').collect()

[Row(h8_30_to_9=891135)]

In [8]:
sct.sc.stop()

# Run Workload

In [ ]:
if database!="":
    load_table=False
    bm.sct.spark.sql("use " + database)
else:
    load_table=True
    if tabledir=="":
        raise "Either database or tabledir should be set"

In [ ]:
!echo "" > /opt/spark/work-dir/telegraf.out

In [ ]:
bm.test_tpc.power_run(explain=False, print_result=False, load_table=load_table, action=lambda df: df.collect())

In [ ]:
bm.test_tpc.print_result()

In [ ]:
#collect_worker=false, jenkins host will do the collection fro workers
bm.collect_profile(False)